# 03 - Tester la performance des fonctions de detection

Ce notebook ne lit pas de PDF. Il sert de banc de test parametrable a partir d'un tableau pandas saisi directement dans le notebook.

Il teste le moteur generique configure par CSV, le moteur Article 9 et le pipeline complet sur texte simule.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from compliance_nlp import (
    load_article9_terms,
    load_generic_detection_rules,
    load_section_definitions,
    load_whitelist_terms,
)
from compliance_nlp.article9 import analyze_article9_section
from compliance_nlp.generic import analyze_generic_section
from compliance_nlp.pipeline import analyze_text

import pandas as pd

## Chargement des referentiels

In [ ]:
sections = load_section_definitions(repo_root / "configs" / "sections.csv")
generic_rules = load_generic_detection_rules(repo_root / "configs" / "generic_detection_rules.csv")
article9_terms = load_article9_terms(repo_root / "configs" / "article9_terms.csv")
whitelist_terms = load_whitelist_terms(repo_root / "configs" / "article9_whitelist.csv")

pd.DataFrame([
    {"referentiel": "sections", "count": len(sections)},
    {"referentiel": "generic_detection_rules", "count": len(generic_rules)},
    {"referentiel": "article9_terms", "count": len(article9_terms)},
    {"referentiel": "article9_whitelist", "count": len(whitelist_terms)},
])

In [ ]:
generic_rules_df = pd.DataFrame([
    {
        "rule_id": rule.rule_id,
        "section_scope": " | ".join(rule.section_scope),
        "category": rule.category,
        "label": rule.label,
        "terms": " | ".join(rule.terms),
        "synonyms": " | ".join(rule.synonyms),
        "alert_level": rule.alert_level,
        "severity": rule.severity,
        "base_score": rule.base_score,
        "fuzzy_threshold": rule.fuzzy_threshold,
    }
    for rule in generic_rules
])
generic_rules_df

In [ ]:
article9_rules_df = pd.DataFrame([
    {
        "rule_id": rule.rule_id,
        "category": rule.category,
        "label": rule.label,
        "terms": " | ".join(rule.terms),
        "synonyms": " | ".join(rule.synonyms),
        "alert_level": rule.alert_level,
        "base_score": rule.base_score,
        "fuzzy_threshold": rule.fuzzy_threshold,
    }
    for rule in article9_terms
])
article9_rules_df

## Tableau des cas de test

Ajoute tes cas directement dans `test_cases`.

- `control_family` : `generic`, `article9` ou `full_text`
- `section_id` : section cible pour `generic` et `article9`
- `text` : texte a tester
- `expected_alert` : alerte attendue oui/non
- `expected_keys` : `rule_id` attendus separes par `|`
- `expected_categories` : categories attendues separees par `|`
- `expected_detection_types` : `exact`, `synonym`, `root`, `fuzzy`, separes par `|`

In [ ]:
test_cases = pd.DataFrame([
    {
        "case_id": "GEN_EXACT_BENEF_PROCHES",
        "control_family": "generic",
        "section_id": "beneficiaires",
        "text": "Je designe mes proches par parts egales.",
        "expected_alert": True,
        "expected_keys": "beneficiary_clause_imprecise|forbidden_ambigue",
        "expected_categories": "clause_beneficiaire|mot_interdit",
        "expected_detection_types": "exact",
        "comment": "Terme exact sur section beneficiaires.",
    },
    {
        "case_id": "GEN_EXACT_COMPOUND_SANS_RISQUE",
        "control_family": "generic",
        "section_id": "conseil",
        "text": "Ce placement est sans risque pour le client.",
        "expected_alert": True,
        "expected_keys": "advice_risk_minimization|forbidden_sans_risque",
        "expected_categories": "conseil|mot_interdit",
        "expected_detection_types": "exact|synonym",
        "comment": "Mot compose et synonyme configure.",
    },
    {
        "case_id": "GEN_SYNONYM_FONCER",
        "control_family": "generic",
        "section_id": "conseil",
        "text": "Le conseiller indique qu'il faut foncer.",
        "expected_alert": True,
        "expected_keys": "advice_unprofessional_wording|forbidden_alerte",
        "expected_categories": "conseil|mot_interdit",
        "expected_detection_types": "synonym|exact",
        "comment": "Synonyme configure pour conseil + mot surveille.",
    },
    {
        "case_id": "GEN_ROOT_GARANTIS",
        "control_family": "generic",
        "section_id": "conseil",
        "text": "Les rendements sont garantis par la compagnie.",
        "expected_alert": True,
        "expected_keys": "advice_risk_minimization|forbidden_garanti",
        "expected_categories": "conseil|mot_interdit",
        "expected_detection_types": "root|fuzzy",
        "comment": "Variante racine/pluriel autour de garanti.",
    },
    {
        "case_id": "GEN_FUZZY_GARENTIE",
        "control_family": "generic",
        "section_id": "conseil",
        "text": "Le capital est garentie par le contrat.",
        "expected_alert": True,
        "expected_keys": "advice_risk_minimization",
        "expected_categories": "conseil|mot_interdit",
        "expected_detection_types": "fuzzy",
        "comment": "Faute volontaire sur garantie.",
    },
    {
        "case_id": "GEN_WRONG_SECTION_NO_ALERT",
        "control_family": "generic",
        "section_id": "beneficiaires",
        "text": "Les risques ne sont pas un vrai sujet ici.",
        "expected_alert": False,
        "expected_keys": "",
        "expected_categories": "",
        "expected_detection_types": "",
        "comment": "Regle conseil presente mais section beneficiaires.",
    },
    {
        "case_id": "A9_EXACT_DIABETIQUE",
        "control_family": "article9",
        "section_id": "document",
        "text": "Le client indique etre diabetique depuis plusieurs annees.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_categories": "sante",
        "expected_detection_types": "exact",
        "comment": "Terme exact Article 9.",
    },
    {
        "case_id": "A9_SYNONYM_INSULINE",
        "control_family": "article9",
        "section_id": "document",
        "text": "Observation libre : traitement par insuline signale par l'adherent.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_categories": "sante",
        "expected_detection_types": "synonym",
        "comment": "Synonyme Article 9.",
    },
    {
        "case_id": "A9_ROOT_DIABETIQUES",
        "control_family": "article9",
        "section_id": "document",
        "text": "Les informations indiquent des antecedents diabetiques.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_categories": "sante",
        "expected_detection_types": "root|exact",
        "comment": "Variante autour de diabetique.",
    },
    {
        "case_id": "A9_FUZZY_DIABETTIQUE",
        "control_family": "article9",
        "section_id": "document",
        "text": "Le souscripteur precise etre diabettique.",
        "expected_alert": True,
        "expected_keys": "SANTE_DIABETE",
        "expected_categories": "sante",
        "expected_detection_types": "fuzzy",
        "comment": "Faute volontaire Article 9.",
    },
    {
        "case_id": "A9_WHITELIST_NEGATIVE",
        "control_family": "article9",
        "section_id": "document",
        "text": "La sante financiere du contrat est correcte.",
        "expected_alert": False,
        "expected_keys": "",
        "expected_categories": "",
        "expected_detection_types": "",
        "comment": "Expression attendue en whitelist.",
    },
    {
        "case_id": "FULL_TEXT_GENERIC_AND_A9",
        "control_family": "full_text",
        "section_id": "document",
        "text": """
        7. Beneficiaires
        Mes proches par parts egales.
        8. Declarations
        Le client indique etre diabetique.
        9. Conseil et Recommandation
        Ce placement est sans risque.
        10. Signatures
        """,
        "expected_alert": True,
        "expected_keys": "beneficiary_clause_imprecise|forbidden_ambigue|advice_risk_minimization|forbidden_sans_risque|SANTE_DIABETE",
        "expected_categories": "clause_beneficiaire|mot_interdit|sante",
        "expected_detection_types": "exact|synonym",
        "comment": "Pipeline complet avec sections + Article 9.",
    },
    {
        "case_id": "NEGATIVE_NEUTRAL",
        "control_family": "generic",
        "section_id": "conseil",
        "text": "Le client souhaite modifier son adresse postale.",
        "expected_alert": False,
        "expected_keys": "",
        "expected_categories": "",
        "expected_detection_types": "",
        "comment": "Phrase neutre.",
    },
])

test_cases

## Fonctions d'execution et d'evaluation

In [ ]:
def split_expected(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {item.strip() for item in str(value).split("|") if item.strip()}


def finding_key(finding):
    return finding.rule_id or finding.code


def run_detector(row):
    family = row["control_family"]
    text = row["text"]
    section_id = row.get("section_id") or "document"

    if family == "generic":
        return analyze_generic_section(section_id, text, generic_rules)
    if family == "article9":
        return analyze_article9_section(section_id, text, article9_terms, whitelist_terms)
    if family == "full_text":
        return analyze_text(
            document_name=row["case_id"],
            source_path="notebook",
            extracted_text=text,
            generic_rules=generic_rules,
            section_definitions=sections,
            article9_terms=article9_terms,
            whitelist_terms=whitelist_terms,
        ).findings

    raise ValueError(f"control_family inconnu: {family}")


def set_to_text(values):
    return " | ".join(sorted(value for value in values if value))


def findings_to_rows(case, findings):
    if not findings:
        return [{
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "section_id": case["section_id"],
            "predicted_key": None,
            "code": None,
            "rule_id": None,
            "category": None,
            "matched_term": None,
            "detection_type": None,
            "score": None,
            "severity": None,
            "evidence": None,
        }]

    return [
        {
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "section_id": case["section_id"],
            "predicted_key": finding_key(finding),
            "code": finding.code,
            "rule_id": finding.rule_id,
            "category": finding.category,
            "matched_term": finding.matched_term,
            "detection_type": finding.detection_type,
            "score": finding.score,
            "severity": finding.severity,
            "evidence": finding.evidence,
        }
        for finding in findings
    ]

## Lancement du banc de test

In [ ]:
case_results = []
finding_rows = []

for _, case in test_cases.iterrows():
    findings = run_detector(case)
    predicted_keys = {finding_key(finding) for finding in findings}
    predicted_categories = {finding.category for finding in findings if finding.category}
    predicted_detection_types = {finding.detection_type for finding in findings if finding.detection_type}

    expected_keys = split_expected(case["expected_keys"])
    expected_categories = split_expected(case["expected_categories"])
    expected_detection_types = split_expected(case["expected_detection_types"])
    expected_alert = bool(case["expected_alert"])
    detected = bool(findings)

    if expected_alert and detected:
        alert_outcome = "TP"
    elif expected_alert and not detected:
        alert_outcome = "FN"
    elif not expected_alert and detected:
        alert_outcome = "FP"
    else:
        alert_outcome = "TN"

    expected_key_match = expected_keys.issubset(predicted_keys)
    unexpected_keys = predicted_keys - expected_keys if expected_keys else predicted_keys if not expected_alert else set()
    missing_keys = expected_keys - predicted_keys
    category_match = not expected_categories or bool(expected_categories & predicted_categories)
    detection_type_match = not expected_detection_types or bool(expected_detection_types & predicted_detection_types)
    max_score = max([finding.score for finding in findings if finding.score is not None], default=None)

    case_results.append({
        **case.to_dict(),
        "detected": detected,
        "predicted_keys": set_to_text(predicted_keys),
        "missing_keys": set_to_text(missing_keys),
        "unexpected_keys": set_to_text(unexpected_keys),
        "predicted_categories": set_to_text(predicted_categories),
        "predicted_detection_types": set_to_text(predicted_detection_types),
        "finding_count": len(findings),
        "max_score": max_score,
        "alert_outcome": alert_outcome,
        "expected_key_match": expected_key_match,
        "category_match": category_match,
        "detection_type_match": detection_type_match,
        "case_ok": (
            alert_outcome in {"TP", "TN"}
            and expected_key_match
            and category_match
            and detection_type_match
            and not unexpected_keys
        ),
    })
    finding_rows.extend(findings_to_rows(case, findings))

case_results_df = pd.DataFrame(case_results)
findings_df = pd.DataFrame(finding_rows)

case_results_df

## Synthese qualite des cas

In [ ]:
case_results_df.groupby(["control_family", "alert_outcome"], dropna=False).size().reset_index(name="count")

In [ ]:
quality_summary = pd.DataFrame([{
    "case_count": len(case_results_df),
    "ok_count": int(case_results_df["case_ok"].sum()),
    "ko_count": int((~case_results_df["case_ok"]).sum()),
    "ok_rate": case_results_df["case_ok"].mean(),
}])
quality_summary

## Mesures alerte TP / FP / FN / TN

In [ ]:
tp = (case_results_df["alert_outcome"] == "TP").sum()
fp = (case_results_df["alert_outcome"] == "FP").sum()
fn = (case_results_df["alert_outcome"] == "FN").sum()
tn = (case_results_df["alert_outcome"] == "TN").sum()

precision = tp / (tp + fp) if (tp + fp) else None
recall = tp / (tp + fn) if (tp + fn) else None
f1 = 2 * precision * recall / (precision + recall) if precision and recall else None

pd.DataFrame([{
    "TP": tp,
    "FP": fp,
    "FN": fn,
    "TN": tn,
    "precision": precision,
    "recall": recall,
    "f1": f1,
}])

## Cas a corriger ou a investiguer

In [ ]:
case_results_df[~case_results_df["case_ok"]][[
    "case_id",
    "control_family",
    "section_id",
    "text",
    "expected_alert",
    "alert_outcome",
    "expected_keys",
    "predicted_keys",
    "missing_keys",
    "unexpected_keys",
    "expected_detection_types",
    "predicted_detection_types",
    "max_score",
    "comment",
]]

## Detail des findings produits

In [ ]:
findings_df.sort_values(["case_id", "score"], ascending=[True, False], na_position="last")

## Performance par type de detection

In [ ]:
findings_df[findings_df["predicted_key"].notna()].groupby(
    ["control_family", "category", "detection_type"], dropna=False
).agg(
    count=("predicted_key", "size"),
    avg_score=("score", "mean"),
    min_score=("score", "min"),
    max_score=("score", "max"),
).reset_index().sort_values(["control_family", "category", "detection_type"])

## Simulation des seuils de score

In [ ]:
threshold_rows = []

for threshold in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    evaluated = case_results_df.copy()
    evaluated["detected_at_threshold"] = evaluated["max_score"].fillna(0) >= threshold

    tp_t = ((evaluated["expected_alert"]) & (evaluated["detected_at_threshold"])).sum()
    fp_t = ((~evaluated["expected_alert"]) & (evaluated["detected_at_threshold"])).sum()
    fn_t = ((evaluated["expected_alert"]) & (~evaluated["detected_at_threshold"])).sum()
    tn_t = ((~evaluated["expected_alert"]) & (~evaluated["detected_at_threshold"])).sum()
    precision_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) else None
    recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else None
    f1_t = 2 * precision_t * recall_t / (precision_t + recall_t) if precision_t and recall_t else None

    threshold_rows.append({
        "threshold": threshold,
        "TP": tp_t,
        "FP": fp_t,
        "FN": fn_t,
        "TN": tn_t,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t,
    })

thresholds_df = pd.DataFrame(threshold_rows)
thresholds_df

## Export des resultats du banc de test

In [ ]:
export_dir = repo_root / "outputs" / "performance"
export_dir.mkdir(parents=True, exist_ok=True)

case_results_path = export_dir / "detection_case_results.csv"
findings_path = export_dir / "detection_findings.csv"
thresholds_path = export_dir / "detection_thresholds.csv"

case_results_df.to_csv(case_results_path, index=False, encoding="utf-8-sig")
findings_df.to_csv(findings_path, index=False, encoding="utf-8-sig")
thresholds_df.to_csv(thresholds_path, index=False, encoding="utf-8-sig")

{
    "case_results": str(case_results_path),
    "findings": str(findings_path),
    "thresholds": str(thresholds_path),
}